# Notebook 03 — Cluster Analysis

**Goal:** Identify user segments using KMeans and HDBSCAN on the behavioural feature matrix. Select the best model via silhouette/elbow analysis, evaluate cluster quality, and characterise each segment.

**Inputs:** `data/processed/user_features.parquet`

**Outputs:**
- `data/processed/cluster_labels.parquet` — user IDs with cluster assignments
- `outputs/figures/elbow.html` — KMeans model selection chart
- `outputs/figures/umap_clusters.html` — 2D UMAP scatter
- `outputs/figures/cluster_heatmap.html` — feature profile heatmap

In [ ]:
import sys
import logging
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(name)s: %(message)s')

import pandas as pd
import numpy as np

Path('../outputs/figures').mkdir(parents=True, exist_ok=True)

In [ ]:
from src.data.loader import load_config

cfg = load_config('../configs/config.yaml')
cluster_cfg = cfg['clustering']

feature_matrix = pd.read_parquet('../data/processed/user_features.parquet')
print(f'Feature matrix: {feature_matrix.shape[0]} users × {feature_matrix.shape[1]} features')

## 1. KMeans — Elbow & Silhouette Analysis

Sweep k from 3 to 10. PCA is applied first to remove multicollinearity.

In [ ]:
from src.clustering.pipeline import run_clustering_pipeline
from src.clustering.evaluation import elbow_data
from src.visualization.plots import plot_elbow_inertia, plot_elbow_silhouette, save_figure

# Run KMeans sweep
kmeans_result = run_clustering_pipeline(
    feature_matrix=feature_matrix,
    algorithm='kmeans',
    use_pca=True,
    use_umap_viz=True,
    config=cluster_cfg,
)

print(f'Best k: {kmeans_result.n_clusters}')
print(f'Silhouette: {kmeans_result.silhouette:.4f}')
print(f'Davies-Bouldin: {kmeans_result.davies_bouldin:.4f}')
print(f'Calinski-Harabasz: {kmeans_result.calinski_harabasz:.1f}')

In [ ]:
# The pipeline already did the sweep internally;
# re-run sweep explicitly to capture all inertia/silhouette values for elbow plot
from src.clustering.pipeline import preprocess, reduce_with_pca, cluster_kmeans

X_scaled, feature_names = preprocess(feature_matrix)
X_pca, pca_obj = reduce_with_pca(X_scaled)

k_min = cluster_cfg['kmeans']['n_clusters_range'][0]
k_max = cluster_cfg['kmeans']['n_clusters_range'][1]

_, best_k, sil_scores, inertias = cluster_kmeans(
    X_pca,
    k_range=(k_min, k_max),
    n_init=cluster_cfg['kmeans']['n_init'],
    random_state=cluster_cfg['kmeans']['random_state'],
)

elbow_df = elbow_data(inertias, sil_scores)
fig_elbow_inertia = plot_elbow_inertia(elbow_df)
fig_elbow_silhouette = plot_elbow_silhouette(elbow_df)
save_figure(fig_elbow_inertia, '../outputs/figures/elbow_inertia')
save_figure(fig_elbow_silhouette, '../outputs/figures/elbow_silhouette')
fig_elbow_inertia.show()
fig_elbow_silhouette.show()


## 2. HDBSCAN — Density-Based Clustering

In [ ]:
hdbscan_result = run_clustering_pipeline(
    feature_matrix=feature_matrix,
    algorithm='hdbscan',
    use_pca=True,
    use_umap_viz=True,
    config=cluster_cfg,
)

print(f'HDBSCAN clusters: {hdbscan_result.n_clusters}')
print(f'Noise points: {(hdbscan_result.labels == -1).sum()}')
print(f'Silhouette: {hdbscan_result.silhouette:.4f}')

## 3. GMM, Agglomerative & Spectral Clustering

In [ ]:
from src.clustering.pipeline import cluster_gmm, cluster_agglomerative, cluster_minibatch_kmeans, within_cluster_rmse
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

k_min = cluster_cfg['kmeans']['n_clusters_range'][0]
k_max = cluster_cfg['kmeans']['n_clusters_range'][1]
best_k = kmeans_result.n_clusters

# GMM — sweep k via BIC
gmm_labels, gmm_k, gmm_bic, gmm_aic = cluster_gmm(X_pca, k_range=(k_min, k_max))
print(f'GMM best k={gmm_k} (BIC={gmm_bic:.1f}, AIC={gmm_aic:.1f})')

# Agglomerative — use same k as best KMeans for fair comparison
agg_labels = cluster_agglomerative(X_pca, n_clusters=best_k)
print(f'Agglomerative: k={best_k}')

# MiniBatchKMeans — faster stochastic variant of KMeans
mb_labels, mb_k = cluster_minibatch_kmeans(X_pca, k_range=(k_min, k_max))
print(f'MiniBatchKMeans best k={mb_k}')


## 4. Algorithm Comparison Table

Green = best value in each column. Silhouette ↑ and Calinski-Harabasz ↑ (higher is better); Davies-Bouldin ↓ and Within-Cluster RMSE ↓ (lower is better).

In [ ]:
from src.visualization.plots import plot_comparison_table, save_figure

def _metrics(X, labels, name):
    mask = labels != -1
    Xm, ym = X[mask], labels[mask]
    n_clusters = len(set(ym))
    if n_clusters < 2:
        return {}
    return {
        'Algorithm': name,
        'Clusters': n_clusters,
        'Noise Points': int((labels == -1).sum()),
        'Silhouette ↑': round(silhouette_score(Xm, ym), 4),
        'Davies-Bouldin ↓': round(davies_bouldin_score(Xm, ym), 4),
        'Calinski-Harabasz ↑': round(calinski_harabasz_score(Xm, ym), 1),
        'Within-Cluster RMSE ↓': round(within_cluster_rmse(Xm, ym), 4),
    }

rows = [
    _metrics(X_pca, kmeans_result.labels,  f'KMeans (k={kmeans_result.n_clusters})'),
    _metrics(X_pca, hdbscan_result.labels, f'HDBSCAN ({hdbscan_result.n_clusters} clusters)'),
    _metrics(X_pca, gmm_labels,            f'GMM (k={gmm_k})'),
    _metrics(X_pca, agg_labels,            f'Agglomerative (k={best_k})'),
    _metrics(X_pca, mb_labels,           f'MiniBatchKMeans (k={mb_k})'),
]
comparison = pd.DataFrame([r for r in rows if r]).set_index('Algorithm')

fig_table = plot_comparison_table(comparison.reset_index())
save_figure(fig_table, '../outputs/figures/algorithm_comparison')
fig_table.show()
print(comparison.to_string())


In [ ]:
# Select best result across all algorithms by silhouette score
all_results = [
    (kmeans_result.silhouette,  kmeans_result,  kmeans_result.labels,  f'KMeans k={kmeans_result.n_clusters}'),
    (hdbscan_result.silhouette, hdbscan_result, hdbscan_result.labels, f'HDBSCAN {hdbscan_result.n_clusters} clusters'),
]

# Compute silhouette for new algorithms
for name, labs in [('GMM', gmm_labels), ('Agglomerative', agg_labels), ('Spectral', mb_labels)]:
    mask = labs != -1
    if mask.sum() > 1 and len(set(labs[mask])) > 1:
        sil = silhouette_score(X_pca[mask], labs[mask])
        all_results.append((sil, None, labs, name))

all_results.sort(key=lambda x: x[0], reverse=True)
best_sil, best_result_obj, best_labels_arr, best_name = all_results[0]

# Use ClusterResult from the pipeline if available, else fall back
best_result = best_result_obj if best_result_obj is not None else kmeans_result
if best_result_obj is None:
    best_result.labels = best_labels_arr

print(f'Selected: {best_name} (silhouette={best_sil:.4f})')
print('\nRanking:')
for sil, _, _, name in all_results:
    marker = ' <-- BEST' if name == best_name else ''
    print(f'  {name:<30} silhouette={sil:.4f}{marker}')
labels = best_labels_arr


## 5. Cluster Stability (Bootstrap Silhouette)

In [ ]:
from src.clustering.evaluation import bootstrap_silhouette

mean_sil, std_sil = bootstrap_silhouette(
    best_result.feature_matrix_scaled,
    best_result.labels,
    n_bootstrap=50,
)
print(f'Bootstrap silhouette: {mean_sil:.4f} ± {std_sil:.4f}')

## 6. Cluster Profiling

In [ ]:
from src.clustering.evaluation import summarise_clusters, label_clusters, feature_importance

cluster_summary = summarise_clusters(feature_matrix, best_result.labels)
cluster_names = label_clusters(cluster_summary, feature_matrix, best_result.labels)

print('Cluster labels:')
for cid, name in cluster_names.items():
    n = (best_result.labels == cid).sum()
    print(f'  Cluster {cid} ({n} users): {name}')

In [ ]:
# Feature importance across clusters
imp_df = feature_importance(feature_matrix, best_result.labels)
print('Top 10 most discriminating features:')
print(imp_df.head(10).to_string(index=False))

## 7. Save Cluster Labels

In [ ]:
labels_df = pd.DataFrame({
    'userid': feature_matrix.index,
    'cluster': best_result.labels,
    'cluster_name': [cluster_names.get(c, str(c)) for c in best_result.labels],
})

if best_result.umap_coords is not None:
    labels_df['umap_x'] = best_result.umap_coords[:, 0]
    labels_df['umap_y'] = best_result.umap_coords[:, 1]

labels_df.to_parquet('../data/processed/cluster_labels.parquet', index=False)
print(f'Cluster labels saved: {labels_df.shape}')
labels_df['cluster_name'].value_counts()

Proceed to **Notebook 04** for interactive visualisations and business insights.